In [61]:
!pip install scikit-learn pandas numpy matplotlib seaborn pymorphy3 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 61.0 MB/s eta 0:00:00


In [62]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as snsd
import warnings
import nltk
import joblib

from pymorphy3 import MorphAnalyzer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.decomposition import PCA

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

warnings.filterwarnings("ignore", category=FutureWarning)

In [64]:
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
stop_words = joblib.load('stop_words.joblib')

morph = MorphAnalyzer()

In [56]:
df = pd.read_csv('unlabeled_df.csv')
df.shape

(12418, 2)

In [57]:
df = df.dropna(subset=['content'])
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12395 entries, 0 to 12417
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   content  12395 non-null  object 
 1   topic    0 non-null      float64
dtypes: float64(1), object(1)
memory usage: 290.5+ KB


In [65]:
def clean_text(text):

    text = str(text).lower() # нижний регистр

    text = re.sub(r'<[^>]+>', '', text) # чистим от html-тегов
    text = re.sub(r'https?://\S+|www\.\S+', '', text) # чистим от url
    text = re.sub(r'[^а-яё\s]', '', text) # убираем знаки и цифры
    text = re.sub(r'\s+', ' ', text).strip() # убираем лишние пробелы

    return text

In [66]:
def preprocess_text(text, morph=morph, stop_words=stop_words):
    """
    Полная предобработка текста: очистка -> токенизация -> лемматизация -> фильтрация
    """

    text = clean_text(text)

    tokens = word_tokenize(text)
    tokens = [morph.parse(token)[0].normal_form for token in tokens]
    tokens = [token for token in tokens if token not in stop_words and len(token) > 3]

    return ' '.join(tokens)

In [ ]:

df['content'] = df['content'].apply(preprocess_text)

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=5000,           # максимальное количество признаков
    max_df=0.8,                  # игнорировать слова, встречающиеся более чем в 80% документов
    min_df=5,                    # игнорировать слова, встречающиеся менее чем в 5 документах
    ngram_range=(1, 2)         # использовать униграммы и биграммы
)

In [ ]:
tfidf_matrix = vectorizer.fit_transform(df['content'])
print(f"Размер TF-IDF матрицы: {tfidf_matrix.shape}")

In [ ]:
print("\nОпределение оптимального количества кластеров...")
inertias = []
K_range = range(5, 31, 5)

for k in K_range:
    kmeans_temp = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans_temp.fit(tfidf_matrix)
    inertias.append(kmeans_temp.inertia_)

# Визуализация метода локтя
plt.figure(figsize=(10, 6))
plt.plot(K_range, inertias, 'bo-')
plt.xlabel('Количество кластеров')
plt.ylabel('Inertia')
plt.title('Метод локтя для определения оптимального количества кластеров')
plt.grid(True)
plt.show()

In [ ]:
num_clusters = 5  # измените это значение на основе графика выше

print(f"\nКластеризация на {num_clusters} кластеров...")
kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
clusters = kmeans.fit_predict(tfidf_matrix)

In [ ]:
df['cluster'] = clusters

In [ ]:
def get_top_keywords_per_cluster(tfidf_matrix, clusters, vectorizer, n_terms=15):
    """
    Извлекает топ-слова для каждого кластера на основе TF-IDF
    """
    feature_names = vectorizer.get_feature_names_out()
    cluster_keywords = {}

    for cluster_id in range(clusters.max() + 1):
        # Индексы документов в кластере
        cluster_indices = np.where(clusters == cluster_id)[0]

        # Средние TF-IDF значения для кластера
        cluster_tfidf = tfidf_matrix[cluster_indices].mean(axis=0)
        cluster_tfidf_array = np.asarray(cluster_tfidf).flatten()

        # Топ-слова по TF-IDF
        top_indices = cluster_tfidf_array.argsort()[-n_terms:][::-1]
        top_terms = [(feature_names[i], cluster_tfidf_array[i]) for i in top_indices]

        cluster_keywords[cluster_id] = top_terms

    return cluster_keywords

In [ ]:
cluster_keywords = get_top_keywords_per_cluster(tfidf_matrix, clusters, vectorizer, n_terms=20)

In [ ]:
for cluster_id, keywords in cluster_keywords.items():
    cluster_size = len(df[df['cluster'] == cluster_id])
    print(f"\nКЛАСТЕР {cluster_id} (статей: {cluster_size}):")
    print("-" * 60)
    for term, score in keywords:
        print(f"  {term:30s} {score:.4f}")